<a href="https://colab.research.google.com/github/Almapian/Ml_hackathon/blob/main/mlds_hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

In [ ]:
# For colab users
drive.mount('/ML')

Drive already mounted at /ML; to attempt to forcibly remount, call drive.mount("/ML", force_remount=True).


In [ ]:
import pandas as pd
import os

# Set data path here here
data_dir = '/ML/My Drive/ML'
# data_dir = './data' # the path to where your data is stored

trips = pd.read_csv(os.path.join(data_dir, 'train_val_trips.csv'))
house = pd.read_csv(os.path.join(data_dir, 'train_val_households.csv'))
person = pd.read_csv(os.path.join(data_dir, 'train_val_persons.csv'))

In [ ]:
trips.head()

,tripid,hhid,persid,tripno,starthour,startime,arrhour,arrtime,origplace1,destplace1,triptime,travtime,cumdist,trippurp,duration,mode
0,0,0,0,1,9,545,9,548,Other place,Outdoor feature,3,3,0.28130,Recreational,41,DRIVE
1,1,0,0,2,9,589,9,591,Outdoor feature,Accommodation,2,2,0.30831,Social,82,DRIVE
2,2,0,0,3,11,673,11,675,Accommodation,Outdoor feature,2,2,0.30831,Personal Business,8,DRIVE
3,3,0,0,4,11,683,11,684,Outdoor feature,Outdoor feature,1,1,NaN,Recreational,61,DRIVE
4,4,0,0,5,12,745,12,747,Outdoor feature,Accommodation,2,2,NaN,Personal Business,44,DRIVE


In [ ]:
house.head()

,hhid,hhsize,dwelltype,owndwell,totalbikes,totalvehs,travmonth,travdow,homelga,youngestgroup_5,aveagegroup_5,oldestgroup_5,hhinc_group,homesubregion_ASGS,homeregion_ASGS
0,0,2,Separate House,Fully Owned,1,1,February,Sunday,Darebin (C),55->59,55->59,55->59,"$6,000-$7,999 ($312,000-$415,999)",MELB - Inner,Greater Melbourne
1,1,2,Separate House,Being Rented,0,0,March,Thursday,Wyndham (C),25->29,25->29,25->29,"$2,500-$2,999 ($130,000-$155,999)",MELB - Outer,Greater Melbourne
2,2,1,"Unit, Flat or Apartment",Being Rented,0,1,August,Friday,Mornington Peninsula (S),65->69,65->69,65->69,"$400-$499 ($20,800-$25,999)",MELB - Outer,Greater Melbourne
3,3,3,Separate House,Being Purchased,3,3,October,Friday,Whittlesea (C),15->19,35->39,50->54,"$1,750-$1,999 ($91,000-$103,999)",MELB - Outer,Greater Melbourne
4,4,2,"Unit, Flat or Apartment",Being Rented,0,1,September,Wednesday,Maribyrnong (C),40->44,45->49,50->54,"$2,000-$2,499 ($104,000-$129,999)",MELB - Middle,Greater Melbourne


In [ ]:
person.head()

,persid,hhid,persno,travdow,agegroup,sex,relationship,carlicence,mbikelicence,otherlicence,...,anywfh,wfhmon,wfhtue,wfhwed,wfhthu,wfhfri,wfhsat,wfhsun,wfhtravday,dayType
0,0,0,1,Sunday,55->59,Male,Self,Full Licence,Yes,No,...,Yes,Yes,Yes,No,Yes,Yes,No,No,No,Weekend
1,1,0,2,Sunday,55->59,Female,Spouse,Full Licence,No,No,...,Yes,Yes,Yes,No,No,Yes,No,No,No,Weekend
2,2,1,1,Thursday,25->29,Male,Self,Full Licence,No,No,...,No,No,No,No,No,No,No,No,No,Weekday
3,3,1,2,Thursday,25->29,Female,Spouse,Full Licence,No,No,...,No,No,No,No,No,No,No,No,No,Weekday
4,4,2,1,Friday,65->69,Female,Self,Full Licence,No,No,...,Not in Work Force,Not in Work Force,Not in Work Force,Not in Work Force,Not in Work Force,Not in Work Force,Not in Work Force,Not in Work Force,Not in Work Force,Weekday


In [ ]:
# trips is the base (one row = one trip = one prediction)
df = trips.merge(person, on='persid', how='left', suffixes=('', '_pers'))
df = df.merge(house, on='hhid', how='left', suffixes=('', '_hh'))

# Sanity check
print(df.shape)
print(df['mode'].value_counts())  # check class distribution

(13641, 63)
mode
DRIVE              7194
PASSENGER          3229
WALK               2266
PUBLICTRANSPORT     624
CYCLE               250
OTHER                78
Name: count, dtype: int64


In [ ]:
df.head()

,tripid,hhid,persid,tripno,starthour,startime,arrhour,arrtime,origplace1,destplace1,...,totalvehs,travmonth,travdow_hh,homelga,youngestgroup_5,aveagegroup_5,oldestgroup_5,hhinc_group,homesubregion_ASGS,homeregion_ASGS
0,0,0,0,1,9,545,9,548,Other place,Outdoor feature,...,1,February,Sunday,Darebin (C),55->59,55->59,55->59,"$6,000-$7,999 ($312,000-$415,999)",MELB - Inner,Greater Melbourne
1,1,0,0,2,9,589,9,591,Outdoor feature,Accommodation,...,1,February,Sunday,Darebin (C),55->59,55->59,55->59,"$6,000-$7,999 ($312,000-$415,999)",MELB - Inner,Greater Melbourne
2,2,0,0,3,11,673,11,675,Accommodation,Outdoor feature,...,1,February,Sunday,Darebin (C),55->59,55->59,55->59,"$6,000-$7,999 ($312,000-$415,999)",MELB - Inner,Greater Melbourne
3,3,0,0,4,11,683,11,684,Outdoor feature,Outdoor feature,...,1,February,Sunday,Darebin (C),55->59,55->59,55->59,"$6,000-$7,999 ($312,000-$415,999)",MELB - Inner,Greater Melbourne
4,4,0,0,5,12,745,12,747,Outdoor feature,Accommodation,...,1,February,Sunday,Darebin (C),55->59,55->59,55->59,"$6,000-$7,999 ($312,000-$415,999)",MELB - Inner,Greater Melbourne


In [ ]:
target = 'mode'
#drop_cols = ['tripid', 'hhid', 'persid', 'tripno', 'persno']
drop_cols = ['tripid', 'hhid', 'persid', 'persno', 'wfhmon', 'wfhtue', 'wfhwed',
             'wfhthu', 'wfhfri', 'wfhsat', 'wfhsun', 'homeregion_ASGS']

X = df.drop(columns=drop_cols + [target])
y = df[target]

In [ ]:
numeric_cols = ['cumdist', 'triptime', 'travtime', 'startime', 'arrtime', 'starthour', 'arrhour', 'duration']
for col in numeric_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce')

In [ ]:
X['speed_proxy'] = X['cumdist'] / (X['travtime'] + 1)
X['is_short_trip'] = (X['cumdist'] < 2).astype(int)
X['has_car'] = (X['totalvehs'].astype(str) != '0').astype(int)
X['has_bike'] = (X['totalbikes'].astype(str) != '0').astype(int)
X['is_peak'] = X['starthour'].isin([7, 8, 9, 17, 18, 19]).astype(int)

In [ ]:
# Trip duration in hours (arrtime - startime)
X['trip_duration_mins'] = X['arrtime'] - X['startime']

# Vehicles per person in household
X['vehs_per_person'] = X['totalvehs'] / (X['hhsize'] + 1)

# Is it a work trip?
X['is_work_trip'] = (X['trippurp'] == 'Work Related').astype(int)

# Is destination education?
X['to_education'] = (X['destplace1'] == 'Place of education').astype(int)

X['no_licence'] = (X['carlicence'] == 'No Car Licence').astype(int)  # do before encoding
X['no_licence_long_trip'] = X['no_licence'] * (X['cumdist'] > 5).astype(int)

X['is_walkable'] = (X['cumdist'] < 1).astype(int)    # under 1km, very likely walk
X['is_cycleable'] = (X['cumdist'] < 5).astype(int)   # under 5km, cycling viable
X['is_long_trip'] = (X['cumdist'] > 20).astype(int)  # over 20km, likely drive or PT

In [ ]:
X.head()

,tripno,starthour,startime,arrhour,arrtime,origplace1,destplace1,triptime,travtime,cumdist,...,is_peak,trip_duration_mins,vehs_per_person,is_work_trip,to_education,no_licence,no_licence_long_trip,is_walkable,is_cycleable,is_long_trip
0,1,9,545,9,548,Other place,Outdoor feature,3,3,0.28130,...,1,3,0.333333,0,0,0,0,1,1,0
1,2,9,589,9,591,Outdoor feature,Accommodation,2,2,0.30831,...,1,2,0.333333,0,0,0,0,1,1,0
2,3,11,673,11,675,Accommodation,Outdoor feature,2,2,0.30831,...,0,2,0.333333,0,0,0,0,1,1,0
3,4,11,683,11,684,Outdoor feature,Outdoor feature,1,1,NaN,...,0,1,0.333333,0,0,0,0,0,0,0
4,5,12,745,12,747,Outdoor feature,Accommodation,2,2,NaN,...,0,2,0.333333,0,0,0,0,0,0,0


In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include='object').columns.tolist()

encoders = {}
for col in cat_cols:
    X[col] = X[col].fillna('Missing')
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

X = X.fillna(-999)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
#from sklearn.utils.class_weight import compute_sample_weight

le_target = LabelEncoder()
y_enc = le_target.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

#sample_weights = compute_sample_weight(class_weight='balanced', y=y_enc)

model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=6,
    eval_metric='mlogloss',
    n_estimators=250,
    learning_rate=0.05,       # slower learning
    max_depth=4,              # shallower trees
    subsample=0.75,            # use 80% of rows per tree
    colsample_bytree=0.75,     # use 80% of columns per tree
    min_child_weight=5,       # require more samples per leaf
    reg_lambda=2,             # L2 regularisation
    #early_stopping_rounds=50, # stop if val score doesn't improve
    random_state=42
)
model.fit(X, y_enc)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.75, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=250, n_jobs=None, num_class=6, ...)

In [ ]:
test_trips = pd.read_csv(os.path.join(data_dir, 'test_trips.csv'))
test_persons = pd.read_csv(os.path.join(data_dir, 'test_persons.csv'))
test_house = pd.read_csv(os.path.join(data_dir, 'test_households.csv'))

test_df = test_trips.merge(test_persons, on='persid', how='left', suffixes=('', '_pers'))
test_df = test_df.merge(test_house, on='hhid', how='left', suffixes=('', '_hh'))

trip_ids = test_df['tripid']
X_test = test_df.drop(columns=drop_cols)

# Numeric conversion
for col in numeric_cols:
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# Feature engineering
X_test['speed_proxy'] = X_test['cumdist'] / (X_test['travtime'] + 1)
X_test['is_short_trip'] = (X_test['cumdist'] < 2).astype(int)
X_test['has_car'] = (X_test['totalvehs'].astype(str) != '0').astype(int)
X_test['has_bike'] = (X_test['totalbikes'].astype(str) != '0').astype(int)
X_test['is_peak'] = X_test['starthour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
X_test['trip_duration_mins'] = X_test['arrtime'] - X_test['startime']
X_test['vehs_per_person'] = X_test['totalvehs'] / (X_test['hhsize'] + 1)
X_test['is_work_trip'] = (X_test['trippurp'] == 'Work Related').astype(int)
X_test['to_education'] = (X_test['destplace1'] == 'Place of education').astype(int)
X_test['no_licence'] = (X_test['carlicence'] == 'No Car Licence').astype(int)  # do before encoding
X_test['no_licence_long_trip'] = X_test['no_licence'] * (X_test['cumdist'] > 5).astype(int)
X_test['is_walkable'] = (X_test['cumdist'] < 1).astype(int)    # under 1km, very likely walk
X_test['is_cycleable'] = (X_test['cumdist'] < 5).astype(int)   # under 5km, cycling viable
X_test['is_long_trip'] = (X_test['cumdist'] > 20).astype(int)  # over 20km, likely drive or PT

# Encode using saved encoders (not new ones)
for col in cat_cols:
    X_test[col] = X_test[col].fillna('Missing')
    known = set(encoders[col].classes_)
    # If 'Missing' wasn't in training, replace with the most common value instead
    if 'Missing' not in known:
        X_test[col] = X_test[col].astype(str).apply(lambda x: x if x in known else encoders[col].classes_[0])
    else:
        X_test[col] = X_test[col].astype(str).apply(lambda x: x if x in known else 'Missing')
    X_test[col] = encoders[col].transform(X_test[col])

X_test = X_test.fillna(-999)

print("X columns:", X.shape[1])
print("X_test columns:", X_test.shape[1])
print("Columns in X but not X_test:", set(X.columns) - set(X_test.columns))
print("Columns in X_test but not X:", set(X_test.columns) - set(X.columns))

probs = model.predict_proba(X_test)
class_names = le_target.classes_

submission = pd.DataFrame(probs, columns=class_names)
submission.insert(0, 'tripid', trip_ids.values)
submission.to_csv(f'{data_dir}/submission.csv', index=False)

X columns: 64
X_test columns: 64
Columns in X but not X_test: set()
Columns in X_test but not X: set()
